# 09 · 智能家居实验（Smart Home Lab）

对应官方示例 [Smart Home Demo](https://docs.typesafe.ai/demos/smart-home) 的可运行复刻：3D 仿真屋 + 语音/文字指令 + 真实 Jev 投机提示管线。

**配套内容在文件夹 [`smart_home_demo/`](smart_home_demo/)**：
- `serve_smart_home.py` —— 本地服务（静态页 + Jev/ASR/chat 代理，密钥只走环境变量）
- `smart_home_playground.html` —— 单文件 3D 应用（Three.js + 追踪面板 + 成本统计）

> 上游仓库：[Bald0Wang/jev-playground → smart-home](https://github.com/Bald0Wang/jev-playground/tree/main/smart-home)（含实验报告 README 与演练场笔记本）。

## 1. 官方 Demo 的三个模式

1. **投机提示（Speculative Prompting）**：一次 `/v1/systemone` 调用捆绑全部问题——意图、是否复合、范围、房间、设备类别，**外加五类设备的动作预判**（灯/风扇/音响/家电/门锁，哪怕与本指令无关也问）。代码识别主目标后在代码端剪枝无关分支，零二次往返。
2. **复合拆分**：noul 判定复合（如"关厨房灯然后锁书房门"）→ 交给小模型拆原子指令 → 每条并行再走一次 Jev。
3. **路由守卫**：常识问题（"1989 世界大赛冠军"）由 Jev 快速识别为信息请求，转交通用大模型回答——快而确定的走 Jev，开放式的走 LLM。

我们的复刻在此之上加了**串/并行工作流编排**（noul 判"谁必须先于谁"）与传统 LLM（step-5-preview）同题对照、成本/token 实时统计。

## 2. 启动配套应用

服务默认端口 8810（本格用 8843 独立实例，避免和你正开着的页面打架）。需要 `TYPESAFE_API_KEY`；语音与 LLM 对照另需 `STEPFUN_API_KEY`（阶跃）。

In [1]:
import subprocess, sys, time, json, urllib.request, os
from pathlib import Path

PORT = 8843
BASE = f"http://127.0.0.1:{PORT}"

def up():
    try:
        return urllib.request.urlopen(f"{BASE}/api/health", timeout=2).status == 200
    except Exception:
        return False

if not up():
    env = dict(os.environ)
    env.setdefault("TYPESAFE_API_KEY", os.environ.get("TYPESAFE_API_KEY", ""))
    subprocess.Popen([sys.executable, "serve_smart_home.py", "--port", str(PORT), "--no-open"],
                     cwd=str(Path.cwd() / "smart_home_demo"), env=env,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(20):
        if up(): break
        time.sleep(0.5)

print(json.dumps(json.loads(urllib.request.urlopen(f"{BASE}/api/health", timeout=2).read()),
                 ensure_ascii=False))

{"jev": true, "asr": false, "chat": false, "asr_model": "stepaudio-2.5-asr", "chat_model": "step-1o-turbo-vision", "answer_model": "step-3.5-flash", "compare_model": "step-5-preview"}


## 3. 在 Notebook 里复刻一次投机调用

不看页面也能体验核心机制：把官方式样的**问题包**（意图 + 复合判断 + 设备类别 + 三类动作预判）一次发给 Jev，观察返回的完整概率分布——包括与本指令无关的"投机"答案。

In [2]:
import json, urllib.request

state = {"utterance": "把咖啡烧上",
         "home": {"rooms": ["living_room", "kitchen", "office", "bedroom", "entrance"],
                  "current_state": {"coffee": False, "light_kitchen": True}}}
turn = {"turn_on": "打开/启动", "turn_off": "关闭/停止", "leave_unchanged": "保持现状"}
payload = {"state": state, "model": "jev-latest", "questions": {
    "intent": {"type": "choice", "instructions": "这句指令的意图是什么？", "criteria": {
        "smart_home_command": "控制智能家居设备",
        "information_request": "与智能家居无关的通用知识问题",
        "smart_home_query": "询问设备当前状态"}},
    "is_compound": {"type": "noul", "instructions": "这条指令包含两个或以上互相独立的操作吗？"},
    "category": {"type": "choice", "instructions": "目标设备属于哪一类？", "criteria": {
        "lighting": "灯", "appliance": "家电（如咖啡机）", "lock": "门锁",
        "fan": "风扇", "speaker": "音响", "not_device_specific": "不涉及设备"}},
    "appliance_action": {"type": "choice", "instructions": "投机预判：家电应该怎样？", "criteria": turn},
    "light_action": {"type": "choice", "instructions": "投机预判：灯应该怎样？", "criteria": turn},
    "lock_action": {"type": "choice", "instructions": "投机预判：门锁应该怎样？",
                    "criteria": {"lock": "上锁", "unlock": "解锁", "leave_unchanged": "保持"}},
}}

req = urllib.request.Request(f"{BASE}/api/jev", data=json.dumps(payload).encode(),
                             headers={"Content-Type": "application/json"})
wire = json.loads(urllib.request.urlopen(req, timeout=30).read())
print("上游耗时", wire["upstream_ms"], "ms")
for name, a in wire["body"]["answers"].items():
    dist = a.get("probabilities") or {"noul": a.get("noul")}
    top = max(dist, key=dist.get)
    mark = " ← 采纳" if name in ("intent", "category", "appliance_action") else "（投机，代码端剪枝）"
    print(f"  {name:17s} -> {a.get('choice', a.get('noul'))!s:16s} "
          f"top={top}({dist[top]:.2f}){mark}")

上游耗时 777 ms
  intent            -> smart_home_command top=smart_home_command(1.00) ← 采纳
  is_compound       -> 0.09             top=noul(0.09)（投机，代码端剪枝）
  category          -> appliance        top=appliance(1.00) ← 采纳
  appliance_action  -> turn_on          top=turn_on(1.00) ← 采纳
  light_action      -> leave_unchanged  top=leave_unchanged(0.97)（投机，代码端剪枝）
  lock_action       -> leave_unchanged  top=leave_unchanged(0.97)（投机，代码端剪枝）


**观察与理解：** 与咖啡机无关的 `light_action` / `lock_action` 也返回了完整分布（大概率是"保持现状"或不确定）——这就是投机提示：问都问了，真用到门锁的指令来时不用再跑一趟；本次用不到的，代码端直接丢弃。

## 4. 打开 3D 应用体验

内嵌窗口可直接操作（拖拽旋转 3D、点设备、切管线、双路对比）；麦克风建议在独立标签页体验（部分浏览器限制 iframe 内录音）。

In [3]:
from IPython.display import HTML
HTML(f"""
<div style="border:1px solid #334155;border-radius:12px;overflow:hidden">
  <iframe src="{BASE}/" style="width:100%;height:880px;border:0" allow="microphone"></iframe>
</div>
<p style="font-size:12px;color:#64748b">独立标签页：<a href="{BASE}/" target="_blank">{BASE}</a>
· 语音需 STEPFUN_API_KEY · 无 TYPESAFE_API_KEY 时页面自动用本地模拟引擎</p>""")